# segment-line-intersect-2d — worked example 3: Scan five segments for intersection with one horizontal line

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `segment-line-intersect-2d`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When you need to test many segments against the same infinite line, you can batch all the 2×2 systems into one `(N, 2, 2)` tensor and call `torch.linalg.solve` once. This is far faster than looping in Python. The batched solve returns shape `(N, 2)`, where column 0 is t_seg and column 1 is s_line for each segment.

## Worked solution

**Step 1 — Build the N segment directions.** Stack S0 and S1 arrays and compute `d = S1 - S0`, giving shape (N, 2).

**Step 2 — Broadcast the line direction.** `e = L1 - L0` has shape (2,). We `expand_as(d)` to replicate it N times.

**Step 3 — Build the batched A matrix.** `t.stack([d, -e_expanded], dim=-1)` gives shape (N, 2, 2). Each slice `A[k]` is the 2×2 system for segment k.

**Step 4 — Batched solve.** `t.linalg.solve(A, b)` where `b = L0 - S0` has shape (N, 2). Result has shape (N, 2).

**Step 5 — Extract and hit-test.** `ts[:, 0]` is all t_seg values; `(ts[:, 0] >= 0) & (ts[:, 0] <= 1)` is the boolean hit mask.

In [ ]:
import torch as t

t.manual_seed(0)

# 5 segments, some crossing y=1.5, some not
S0 = t.tensor([[0.0, 0.0], [0.0, 3.0], [-1.0, 0.5], [2.0, 2.0], [0.0, 0.0]], dtype=t.float32)
S1 = t.tensor([[1.0, 3.0], [1.0, 0.0], [ 1.0, 2.0], [3.0, 3.0], [0.5, 1.0]], dtype=t.float32)

# Horizontal infinite line y=1.5: through (0,1.5) and (1,1.5)
L0 = t.tensor([0.0, 1.5])
L1 = t.tensor([1.0, 1.5])

d = S1 - S0                                        # (5, 2)
e = L1 - L0                                        # (2,)
A = t.stack([d, -e.expand_as(d)], dim=-1)          # (5, 2, 2)
b = L0 - S0                                        # (5, 2)
ts = t.linalg.solve(A, b)                          # (5, 2)

t_seg = ts[:, 0]
hit   = (t_seg >= 0.0) & (t_seg <= 1.0)

print('t_seg:', t_seg.tolist())
print('hit  :', hit.tolist())
print(f'{hit.sum().item()} of {len(hit)} segments intersect y=1.5')